# Understanding GRU Networks

This notebook studies Gated Recurrent Units through four small experiments:

1. Implementing a GRU step with NumPy
2. Comparing GRU and LSTM memory mechanisms
3. Tracking gate values over a short sequence
4. Training GRU and LSTM regressors on synthetic sensor data

The examples are designed for learning and experimentation.

## 1. A GRU step implemented with NumPy

In [ ]:

import numpy as np

np.random.seed(27)

def logistic(value):
    clipped = np.clip(value, -500, 500)
    return 1.0 / (1.0 + np.exp(-clipped))


def run_gru_step(current_input, previous_state, parameters):
    """Return the new state and the two GRU gate activations."""
    update = logistic(
        current_input @ parameters["update_input"]
        + previous_state @ parameters["update_hidden"]
        + parameters["update_bias"]
    )

    reset = logistic(
        current_input @ parameters["reset_input"]
        + previous_state @ parameters["reset_hidden"]
        + parameters["reset_bias"]
    )

    candidate = np.tanh(
        current_input @ parameters["candidate_input"]
        + (reset * previous_state) @ parameters["candidate_hidden"]
        + parameters["candidate_bias"]
    )

    new_state = (1.0 - update) * previous_state + update * candidate
    return new_state, update, reset


input_size = 2
state_size = 3

gru_parameters = {
    "update_input": np.random.normal(size=(input_size, state_size)),
    "update_hidden": np.random.normal(size=(state_size, state_size)),
    "update_bias": np.zeros((1, state_size)),
    "reset_input": np.random.normal(size=(input_size, state_size)),
    "reset_hidden": np.random.normal(size=(state_size, state_size)),
    "reset_bias": np.zeros((1, state_size)),
    "candidate_input": np.random.normal(size=(input_size, state_size)),
    "candidate_hidden": np.random.normal(size=(state_size, state_size)),
    "candidate_bias": np.zeros((1, state_size)),
}

example_input = np.array([[0.25, 0.80]])
initial_state = np.zeros((1, state_size))

state, update_gate, reset_gate = run_gru_step(
    example_input,
    initial_state,
    gru_parameters
)

print("New hidden state:", np.round(state, 4))
print("Update gate:", np.round(update_gate, 4))
print("Reset gate:", np.round(reset_gate, 4))


## 2. GRU and LSTM: structural comparison

| Property | LSTM | GRU |
|---|---|---|
| Main gates | Input, forget, output | Update, reset |
| State representation | Cell state plus hidden state | One hidden state |
| Output gate | Present | Not separate |
| Memory update | Uses a separate cell-memory path | Mixes the previous state with a candidate state |
| Parameter count | Usually larger | Usually smaller |
| Typical use | Long or complicated dependencies | Compact sequence models and faster experiments |

The exact parameter count depends on the input size, hidden size, and implementation details.

## 3. Observe GRU gates over a sequence

In [ ]:

import matplotlib.pyplot as plt

# A short normalized sensor sequence.
sensor_values = np.array(
    [0.15, 0.32, 0.28, 0.61, 0.74, 0.55, 0.82, 0.91, 0.68, 0.47, 0.39],
    dtype=np.float32
)

one_feature = 1
hidden_units = 3

np.random.seed(9)
gate_parameters = {
    "update_input": np.random.randn(one_feature, hidden_units),
    "update_hidden": np.random.randn(hidden_units, hidden_units),
    "update_bias": np.zeros((1, hidden_units)),
    "reset_input": np.random.randn(one_feature, hidden_units),
    "reset_hidden": np.random.randn(hidden_units, hidden_units),
    "reset_bias": np.zeros((1, hidden_units)),
    "candidate_input": np.random.randn(one_feature, hidden_units),
    "candidate_hidden": np.random.randn(hidden_units, hidden_units),
    "candidate_bias": np.zeros((1, hidden_units)),
}

state = np.zeros((1, hidden_units))
update_trace = []
reset_trace = []

for value in sensor_values:
    current = np.array([[value]])
    state, update, reset = run_gru_step(current, state, gate_parameters)
    update_trace.append(float(update.mean()))
    reset_trace.append(float(reset.mean()))

time_axis = np.arange(1, len(sensor_values) + 1)

plt.figure(figsize=(9, 4))
plt.plot(time_axis, update_trace, marker="o", label="Average update gate")
plt.plot(time_axis, reset_trace, marker="s", label="Average reset gate")
plt.xlabel("Time step")
plt.ylabel("Average gate activation")
plt.title("GRU Gate Activity")
plt.ylim(0, 1)
plt.xticks(time_axis)
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


## 4. Compare GRU and LSTM on synthetic sensor readings

The target is generated from the recent history of a fictional sensor. This is a demonstration of model construction and timing, not a real forecasting benchmark.

In [ ]:

import time
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.keras.utils.set_random_seed(27)

sample_count = 720
history_length = 10
feature_count = 1

rng = np.random.default_rng(27)
X_data = rng.uniform(
    low=0.0,
    high=1.0,
    size=(sample_count, history_length, feature_count)
).astype("float32")

# Create a target related to the last few observations.
noise = rng.normal(0, 0.03, size=(sample_count, 1)).astype("float32")
y_data = (
    0.55 * X_data[:, -1, 0:1]
    + 0.30 * X_data[:, -2, 0:1]
    + 0.15 * X_data[:, -3, 0:1]
    + noise
).astype("float32")


def build_recurrent_model(kind="gru"):
    recurrent_layer = (
        layers.GRU(20)
        if kind.lower() == "gru"
        else layers.LSTM(20)
    )

    network = keras.Sequential([
        layers.Input(shape=(history_length, feature_count)),
        recurrent_layer,
        layers.Dense(8, activation="relu"),
        layers.Dense(1)
    ], name=f"{kind.lower()}_regressor")

    network.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.0015),
        loss="mse",
        metrics=["mae"]
    )
    return network


gru_network = build_recurrent_model("gru")
lstm_network = build_recurrent_model("lstm")

def train_and_measure(network):
    start = time.perf_counter()
    result = network.fit(
        X_data,
        y_data,
        epochs=6,
        batch_size=24,
        validation_split=0.2,
        verbose=0
    )
    duration = time.perf_counter() - start
    return duration, result.history["loss"][-1], result.history["val_loss"][-1]

gru_duration, gru_train_loss, gru_validation_loss = train_and_measure(gru_network)
lstm_duration, lstm_train_loss, lstm_validation_loss = train_and_measure(lstm_network)

print("GRU")
print(f"  Training time: {gru_duration:.3f} seconds")
print(f"  Final training MSE: {gru_train_loss:.6f}")
print(f"  Final validation MSE: {gru_validation_loss:.6f}")

print("\nLSTM")
print(f"  Training time: {lstm_duration:.3f} seconds")
print(f"  Final training MSE: {lstm_train_loss:.6f}")
print(f"  Final validation MSE: {lstm_validation_loss:.6f}")


## Summary

- A GRU uses update and reset gates to regulate information flow.
- The update gate blends the previous hidden state with a candidate state.
- The reset gate controls how much previous information contributes to the candidate.
- GRUs generally have fewer parameters than comparable LSTMs.
- Runtime and loss depend on the dataset, hardware, initialization, and training settings.
- A fair model comparison should use the same data split, preprocessing, evaluation metric, and repeated trials.